# 05 · Which frauds did the ANN miss?
Audit the held-out replay from notebook 04 at its recorded operating threshold. Compare recall by transaction count with the fraction of fraudulent transaction amount sent for review. Then inspect missed cases by score rank and amount.

This is a retrospective diagnostic on a previously evaluated time window. Use the validation window for subsequent policy design. Transaction amount measures the face value of a payment. Actual losses, recovery rates and investigation costs are unavailable.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/Colab Notebooks/Project 01: Intelligent Financial Fraud Detection")
# Pin the historical replay. Custom batches may have different rows and cutoffs.
batch_folder = "heldout_20261121T181616Z_b3fffdb9"
model_run = "20261121T162317186996Z"
batch_path = project / "Artifacts" / "Batch runs" / batch_folder / "scored_transactions.csv"

Mounted at /content/drive


## Reattach the outcomes after scoring
Notebook 04 stored the original one-based CSV record number. Join by that number because many transactions can have the same time or amount. Verify that the export covers the entire held-out window and uses the saved ANN cutoff.

In [2]:
raw = pd.read_csv(project / "Dataset" / "creditcard.csv", usecols=["Time", "Amount", "Class"])
raw.index = pd.RangeIndex(1, len(raw) + 1, name="source_record")
start = raw.Time.sort_values(kind="stable").iloc[int(.85 * len(raw))]
expected = raw.loc[raw.Time >= start]
scores = pd.read_csv(batch_path, float_precision="round_trip").set_index("source_record", verify_integrity=True)
if not scores.index.sort_values().equals(expected.index.sort_values()):
    raise ValueError("This batch does not contain the complete held-out window.")
aligned = expected.loc[scores.index]
np.testing.assert_array_equal(scores[["Time", "Amount"]], aligned[["Time", "Amount"]])
metrics = pd.read_csv(project / "Artifacts" / f"metrics_{model_run}.csv",
                      float_precision="round_trip").set_index("Model")
cutoff = metrics.loc["ANN", "Threshold"]
if not scores.model_run.eq(model_run).all() or not scores.threshold.eq(cutoff).all():
    raise ValueError("Select the replay at the recorded model run and cutoff.")
if not scores.risk_score.between(0, 1).all() or not aligned.Class.isin([0, 1]).all():
    raise ValueError("Expected finite risk scores in [0,1] and binary labels.")
np.testing.assert_array_equal(scores.send_for_review, scores.risk_score >= cutoff)
audit = scores.assign(is_fraud=aligned.Class.eq(1))
print(f"Matched {len(audit):,} transactions at cutoff {cutoff:.10f}.")

Matched 42,724 transactions at cutoff 0.9998766184.


## Count the four outcomes
A false negative is a fraud that never entered the queue. A false positive is a legitimate transaction the queue sent to a reviewer. Compare their counts with the metrics saved during training.

In [3]:
audit["outcome"] = np.select(
    [audit.is_fraud & audit.send_for_review, audit.is_fraud, audit.send_for_review],
    ["Fraud caught", "Fraud missed", "False alert"], default="Legitimate, unflagged")
summary = audit.groupby("outcome").agg(transactions=("Amount", "size"),
                                       total_amount=("Amount", "sum"))
for outcome, column in [("Fraud caught", "True alerts"), ("Fraud missed", "Missed fraud"),
                        ("False alert", "False alerts")]:
    count = audit.outcome.eq(outcome).sum()
    if count != metrics.loc["ANN", column]:
        raise ValueError(f"{outcome} count differs from the recorded ANN result.")
print(summary.round(2).to_string())

                       transactions  total_amount
outcome                                          
False alert                       3          4.98
Fraud caught                     36       3677.93
Fraud missed                     16       2490.95
Legitimate, unflagged         42669    3093091.51


## Measure how much fraud amount reached review
Count recall gives every fraud equal weight. Amount-weighted recall divides the sum of flagged fraud amounts by the sum of all fraud amounts. A large missed transaction can move the second metric substantially.

In [4]:
frauds = audit.loc[audit.is_fraud]
caught = frauds.loc[frauds.send_for_review]
count_recall = len(caught) / len(frauds)
amount_recall = caught.Amount.sum() / frauds.Amount.sum()
print(f"Frauds flagged: {len(caught)} of {len(frauds)} ({count_recall:.2%})")
print(f"Fraud amount flagged: {caught.Amount.sum():,.2f} of {frauds.Amount.sum():,.2f} ({amount_recall:.2%})")

Frauds flagged: 36 of 52 (69.23%)
Fraud amount flagged: 3,677.93 of 6,168.88 (59.62%)


## Distinguish near misses from low-ranked frauds
Rank all transactions by score. A rank of 40 means the 40th transaction in a descending queue. For tied scores, use the last position in the tie group so that every equal score is treated alike. The missed-case table shows how far a score-based queue would need to extend to include each fraud in this replay.

In [5]:
audit["rank_including_ties"] = audit.risk_score.rank(ascending=False, method="max").astype(int)
misses = audit.loc[audit.outcome.eq("Fraud missed")].sort_values("rank_including_ties")
print(misses[["Amount", "risk_score", "rank_including_ties"]].to_string(float_format=lambda x: f"{x:.6f}"))
largest = misses.nlargest(2, "Amount")
print(f"The two largest misses total {largest.Amount.sum():,.2f} "
      f"({largest.Amount.sum()/misses.Amount.sum():.1%} of missed fraud amount).")

                   Amount  risk_score  rank_including_ties
source_record                                             
280150          77.890000    0.999638                   40
274476          39.900000    0.997409                   47
248972           0.760000    0.995212                   51
272522          12.310000    0.970041                   73
251892           7.060000    0.961831                   78
251882           3.140000    0.717391                  164
249240        1096.990000    0.658507                  184
275993         634.300000    0.572866                  225
245557         357.950000    0.126031                  545
254396           7.590000    0.002042                 3891
254345         187.110000    0.001260                 4860
263081           1.000000    0.000573                 6824
281675          42.530000    0.000572                 6833
245348           2.470000    0.000355                 8312
276072          19.950000    0.000313                 87

## Interpretation for this run
The ANN caught 36 of 52 frauds and sent 59.62% of fraudulent transaction amount to review. One missed fraud was the next transaction below the queue boundary (rank 40). The two largest missed frauds ranked 184th and 225th. Extending the queue by just one review would have caught the nearest miss while leaving those larger transactions outside it.

A follow-up experiment can compare the current score policy with an amount-aware review policy at equal review capacity on validation data. An economic estimate would also need investigation costs and recoverable amounts.